# app17-7 — treinamento do modelo multiclasse (4 estados do motor)

Lê as janelas gravadas no InfluxDB pelo fluxo `FluxoNodeRED-MultiClass` e gera o
`modelo_motor_multiclasse.pkl`, que o `app25` carrega para responder no MQTT.

In [ ]:
!pip install -q influxdb-client pandas scikit-learn matplotlib joblib

## 1) Ler as janelas do InfluxDB

Preencha com os seus valores — os mesmos do nó InfluxDB do Node-RED.

In [ ]:
from influxdb_client import InfluxDBClient
import pandas as pd

INFLUX_URL    = "https://us-east-1-1.aws.cloud2.influxdata.com"
INFLUX_TOKEN  = "SEU_TOKEN_INFLUX_CLOUD"
INFLUX_ORG    = "SUA_ORG"
INFLUX_BUCKET = "IoTSensores"
MEASUREMENT   = "vibracao_multiclasse"

FEATURES = ["mean_ax", "mean_ay", "mean_az",
            "std_ax", "std_ay", "std_az",
            "std_mag", "p2p_mag"]
CLASSES  = ["operando", "inclinado_frente", "inclinado_tras", "anomalia"]

client = InfluxDBClient(url=INFLUX_URL, token=INFLUX_TOKEN, org=INFLUX_ORG)

colunas = ", ".join(f'"{f}"' for f in FEATURES)
flux = f"""
from(bucket: "{INFLUX_BUCKET}")
  |> range(start: -30d)
  |> filter(fn: (r) => r._measurement == "{MEASUREMENT}")
  |> pivot(rowKey: ["_time"], columnKey: ["_field"], valueColumn: "_value")
  |> keep(columns: ["_time", "label", "rodada", {colunas}])
  |> sort(columns: ["_time"])
"""

df = client.query_api().query_data_frame(flux)
if isinstance(df, list):
    df = pd.concat(df, ignore_index=True)

df = (df.rename(columns={"_time": "time"})
        .query("label in @CLASSES")
        .dropna(subset=FEATURES)
        .sort_values("time")
        .reset_index(drop=True))
df["rodada"] = df["rodada"].astype(int)

print(f"{len(df)} janelas")
print(df.groupby(["label", "rodada"]).size().to_string())

## 2) Split por rodada — a última rodada fica de fora do treino

In [ ]:
rodada_teste = df["rodada"].max()

treino = df[df["rodada"] != rodada_teste]
teste  = df[df["rodada"] == rodada_teste]
print(f"treino: {len(treino)} janelas | teste: rodada {rodada_teste}, {len(teste)} janelas")

## 3) Treinar — StandardScaler + rede neural, num Pipeline só

In [ ]:
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

modelo = make_pipeline(
    StandardScaler(),
    MLPClassifier(hidden_layer_sizes=(16,), max_iter=2000, random_state=42),
)

# y em TEXTO: o predict devolve o nome da classe, e a API nao precisa de mapa.
modelo.fit(treino[FEATURES], treino["label"])
print("classes:", list(modelo.classes_))

## 4) Métricas e matriz de confusão

In [ ]:
from sklearn.metrics import (accuracy_score, f1_score, classification_report,
                             ConfusionMatrixDisplay)
import matplotlib.pyplot as plt

y_pred = modelo.predict(teste[FEATURES])

print("acuracia:", round(accuracy_score(teste["label"], y_pred), 3))
print("f1_macro:", round(f1_score(teste["label"], y_pred, average="macro"), 3))
print(classification_report(teste["label"], y_pred, zero_division=0))

ConfusionMatrixDisplay.from_predictions(teste["label"], y_pred,
                                        labels=list(modelo.classes_), xticks_rotation=45)
plt.tight_layout(); plt.show()

## 5) Validação por rodada (`LeaveOneGroupOut`)

In [ ]:
from sklearn.model_selection import LeaveOneGroupOut, cross_val_score

scores = cross_val_score(modelo, df[FEATURES], df["label"], groups=df["rodada"],
                         cv=LeaveOneGroupOut(), scoring="f1_macro")

for rodada, s in zip(sorted(df["rodada"].unique()), scores):
    print(f"rodada {rodada} fora: f1_macro={s:.3f}")
print(f"media: {scores.mean():.3f} (+-{scores.std():.3f})")

## 6) Importância das features (permutation, no conjunto de teste)

In [ ]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(modelo, teste[FEATURES], teste["label"],
                              n_repeats=20, random_state=42)
imp = pd.Series(perm.importances_mean, index=FEATURES).sort_values()

imp.plot.barh(color="tab:orange")
plt.title("Permutation importance (teste)")
plt.tight_layout(); plt.show()

print(imp.sort_values(ascending=False).round(3).to_string())

## 7) Salvar o `.pkl`

In [ ]:
import joblib

ARQUIVO = "modelo_motor_multiclasse.pkl"
joblib.dump(modelo, ARQUIVO)

# Confere recarregando, que e exatamente o que a API faz na inicializacao.
recarregado = joblib.load(ARQUIVO)
print(recarregado.predict(teste[FEATURES].head(3)))

import sklearn
print("scikit-learn:", sklearn.__version__)

try:
    from google.colab import files
    files.download(ARQUIVO)
except Exception:
    print(f"{ARQUIVO} gerado na pasta atual.")

Copie o `.pkl` para `app25-Inferencia-AI-API_Sinais-MultiClass_IMU/api/`, substituindo o
sintético, e acerte o pin do `scikit-learn` no `requirements.txt` para a versão impressa acima.